# 🏗️ Day 02a: REST API Design — Building APIs That Don't Suck

---

## 🎯 What You'll Master Today
- REST constraints and what makes an API truly RESTful
- Resource naming conventions that interviewers love
- URL structure, versioning, pagination, and filtering
- Design a complete REST API from scratch

---

## 🎭 The Analogy

Think of REST like a well-organized library:

- **Resources** = Books, authors, categories (nouns, not verbs!)
- **URLs** = Shelf labels that tell you exactly where to find things
- **HTTP Methods** = Actions: read a book (GET), add a book (POST), update it (PUT)
- **Stateless** = The librarian doesn't remember you between visits — bring your library card every time

A bad library: "Go to shelf 3, turn left, ask Bob."  
A good library: `/fiction/sci-fi/dune` — you know exactly what you'll find.

---

## 📐 REST Constraints — The 6 Rules

```
╔═══════════════════════════════════════════════════════════════════╗
║  THE 6 REST CONSTRAINTS (Roy Fielding, 2000)                     ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. CLIENT-SERVER         Separation of concerns                 ║
║     → Client handles UI, server handles data/logic               ║
║                                                                   ║
║  2. STATELESS             No session on server                   ║
║     → Every request carries ALL needed info (token, etc.)        ║
║                                                                   ║
║  3. CACHEABLE             Responses indicate cacheability        ║
║     → GET is cacheable, POST is not                              ║
║                                                                   ║
║  4. UNIFORM INTERFACE     Consistent resource-based URLs        ║
║     → /users/1, not /getUserById?id=1                           ║
║                                                                   ║
║  5. LAYERED SYSTEM        Client doesn't know about internals   ║
║     → Could be load balancer → API gateway → service            ║
║                                                                   ║
║  6. CODE ON DEMAND (opt)  Server can send executable code       ║
║     → JavaScript in HTML responses (rarely used in APIs)        ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

> **Interview Tip:** Most interviewers care about stateless, uniform interface, and cacheable. Be ready to explain why stateless matters for scaling.

---

## 🔗 Resource Naming — The #1 Thing Interviewers Judge

### Rules to Memorize

```
╔════════════════════════════════════════════════════════════════════╗
║  RESOURCE NAMING RULES                                            ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  ✅ DO:                          ❌ DON'T:                        ║
║  • Use nouns, not verbs          • /getUsers, /createPost         ║
║  • Use plural nouns              • /user/1 (use /users/1)         ║
║  • Use kebab-case                • /userPosts, /user_posts        ║
║  • Nest for relationships        • /getUserPosts?userId=1         ║
║  • Max 2 levels of nesting       • /a/1/b/2/c/3/d/4              ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
```

### Good vs Bad URLs

```
❌ BAD                              ✅ GOOD
──────────────────────────          ──────────────────────────
GET  /getUser/1                     GET  /users/1
POST /createUser                    POST /users
POST /deleteUser/1                  DELETE /users/1
GET  /getUserPosts/1                GET  /users/1/posts
POST /updateUserEmail               PATCH /users/1
GET  /searchUsersByName?n=John      GET  /users?name=John
```

### Complete URL Pattern

```
GET    /api/v1/users              → List all users
POST   /api/v1/users              → Create a user
GET    /api/v1/users/123          → Get user 123
PUT    /api/v1/users/123          → Replace user 123
PATCH  /api/v1/users/123          → Update user 123 partially
DELETE /api/v1/users/123          → Delete user 123
GET    /api/v1/users/123/posts    → List user 123's posts
POST   /api/v1/users/123/posts    → Create a post for user 123
```

---

## 📄 Pagination — Three Approaches

### 1. Offset-Based (Simple, Most Common)
```
GET /users?page=2&limit=20
GET /users?offset=20&limit=20

Response:
{
  "data": [...],
  "total": 150,
  "page": 2,
  "limit": 20,
  "pages": 8
}

⚠️ Problem: Inserting/deleting items shifts pages → can miss/duplicate items
```

### 2. Cursor-Based (Better for Real-Time Data)
```
GET /users?cursor=eyJpZCI6MjB9&limit=20

Response:
{
  "data": [...],
  "next_cursor": "eyJpZCI6NDB9",
  "has_more": true
}

✅ Stable under insertions/deletions
✅ Used by Twitter, Facebook, Slack APIs
❌ Can't jump to page 5
```

### 3. Keyset-Based (Most Performant)
```
GET /users?after_id=20&limit=20

→ SELECT * FROM users WHERE id > 20 ORDER BY id LIMIT 20

✅ Uses index scan (fast even on millions of rows)
```

> **Interview Tip:** "For a social media feed, I'd use cursor-based pagination because offset pagination causes duplicates when new posts are added."

---

## 🔍 Filtering, Sorting & Searching

```
# Filtering
GET /users?status=active&role=admin
GET /products?min_price=100&max_price=500
GET /orders?created_after=2024-01-01

# Sorting
GET /users?sort=created_at&order=desc
GET /products?sort=-price,name    # - prefix = descending

# Searching
GET /users?search=sarthak         # full-text search
GET /users?q=sarthak              # shorthand

# Field selection (sparse fieldsets)
GET /users?fields=id,name,email   # return only specific fields
```

---

## 🔢 API Versioning — Four Strategies

```
╔═══════════════════════════════════════════════════════════════════╗
║  API VERSIONING STRATEGIES                                       ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. URL PATH (Most Common)                                       ║
║     GET /api/v1/users                                            ║
║     GET /api/v2/users                                            ║
║     ✅ Clear, easy to route  ❌ URL pollution                    ║
║                                                                   ║
║  2. QUERY PARAMETER                                              ║
║     GET /api/users?version=2                                     ║
║     ✅ Easy to add  ❌ Easy to forget                            ║
║                                                                   ║
║  3. HEADER                                                        ║
║     Accept: application/vnd.myapi.v2+json                        ║
║     ✅ Clean URLs  ❌ Hidden, harder to test                     ║
║                                                                   ║
║  4. CONTENT NEGOTIATION                                          ║
║     Accept: application/json; version=2                          ║
║     ✅ RESTful  ❌ Complex                                       ║
║                                                                   ║
║  RECOMMENDATION: URL path for simplicity (/api/v1/...)          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 💻 Design Exercise — Blog API

Let's design a complete REST API for a blog platform. This is exactly what you'd do in an interview.

In [ ]:
# ============================================================
# Blog API Design — Complete Resource Map
# ============================================================
# This is how you'd present it in an interview

api_design = """
BASE URL: /api/v1

USERS:
  GET    /users                  → List users (paginated)
  POST   /users                  → Register new user
  GET    /users/{id}             → Get user profile
  PATCH  /users/{id}             → Update profile
  DELETE /users/{id}             → Delete account

POSTS:
  GET    /posts                  → List all posts (?author=1&tag=python&sort=-created_at)
  POST   /posts                  → Create new post
  GET    /posts/{id}             → Get single post
  PUT    /posts/{id}             → Replace post
  PATCH  /posts/{id}             → Update post fields
  DELETE /posts/{id}             → Delete post

NESTED - User's Posts:
  GET    /users/{id}/posts       → Get user's posts

COMMENTS:
  GET    /posts/{id}/comments    → List post's comments
  POST   /posts/{id}/comments    → Add comment to post
  PATCH  /comments/{id}          → Edit comment
  DELETE /comments/{id}          → Delete comment

TAGS:
  GET    /tags                   → List all tags
  GET    /tags/{slug}/posts      → Posts with this tag

AUTH:
  POST   /auth/login             → Login, get tokens
  POST   /auth/register          → Register
  POST   /auth/refresh           → Refresh access token
  POST   /auth/logout            → Invalidate token

SEARCH:
  GET    /search?q=python&type=posts  → Search across resources
"""
print(api_design)

In [ ]:
# ============================================================
# Response Format — Consistent across ALL endpoints
# ============================================================

# Success Response (single item)
success_single = {
    "status": "success",
    "data": {
        "id": 1,
        "title": "REST API Design",
        "author": {"id": 5, "name": "Sarthak"},
        "created_at": "2024-01-15T10:30:00Z",
        "tags": ["python", "fastapi"]
    }
}

# Success Response (list with pagination)
success_list = {
    "status": "success",
    "data": [{"id": 1}, {"id": 2}],
    "meta": {
        "total": 150,
        "page": 1,
        "limit": 20,
        "pages": 8
    }
}

# Error Response
error_response = {
    "status": "error",
    "error": {
        "code": "VALIDATION_ERROR",
        "message": "Invalid input data",
        "details": [
            {"field": "email", "message": "Invalid email format"},
            {"field": "age", "message": "Must be positive"}
        ]
    }
}

import json
print("=== Single Item ===")
print(json.dumps(success_single, indent=2))
print("\n=== Error ===")
print(json.dumps(error_response, indent=2))

In [ ]:
# ============================================================
# HATEOAS — Hypermedia links (advanced, good to mention)
# ============================================================
# Resources include links to related actions

hateoas_response = {
    "id": 1,
    "title": "REST API Design",
    "author": {"id": 5, "name": "Sarthak"},
    "_links": {
        "self": {"href": "/api/v1/posts/1"},
        "author": {"href": "/api/v1/users/5"},
        "comments": {"href": "/api/v1/posts/1/comments"},
        "update": {"href": "/api/v1/posts/1", "method": "PATCH"},
        "delete": {"href": "/api/v1/posts/1", "method": "DELETE"}
    }
}

# Interview: "HATEOAS lets clients discover available actions 
# from the response itself, reducing coupling."
print(json.dumps(hateoas_response, indent=2))

---

## 🗺️ REST API Design Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  REST API DESIGN PATTERNS                                        ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  URLS: plural nouns, kebab-case, max 2 nesting levels            ║
║  /api/v1/users/{id}/posts  ✅                                    ║
║                                                                   ║
║  PAGINATION: cursor > offset for real-time data                  ║
║  VERSIONING: /api/v1/... (URL path is simplest)                  ║
║  FILTERING: query params (?status=active&sort=-created_at)       ║
║                                                                   ║
║  RESPONSE FORMAT: consistent {status, data, meta/error}          ║
║  ERRORS: machine-readable code + human message + field details   ║
║                                                                   ║
║  STATELESS: every request carries auth (JWT in header)           ║
║  AUTH ENDPOINTS: /auth/login — exception to noun rule            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions — Practice These

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What makes an API RESTful? | Stateless, uniform interface, resource-based URLs, proper HTTP methods |
| 2 | Design a URL for getting a user's orders | `GET /users/{id}/orders` — nested resource |
| 3 | Offset vs cursor pagination? | Offset: simple, skip issues. Cursor: stable, performant, can't jump |
| 4 | How to version an API? | URL path (/v1/), query param, header — URL path most common |
| 5 | What is HATEOAS? | Responses include links to discover related actions |
| 6 | Why stateless? | Enables horizontal scaling — any server can handle any request |
| 7 | PUT vs POST for creation? | POST: server decides URL. PUT: client decides URL |
| 8 | How to handle errors? | Consistent format: error code + message + field-level details |
| 9 | Rate limiting in REST? | Return 429, include Retry-After header, X-RateLimit-Remaining |
| 10 | REST vs GraphQL? | REST: fixed responses, multiple endpoints. GraphQL: flexible queries, single endpoint |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Resources are nouns (/users), not verbs (/getUser)   │
## │  • Use HTTP methods for actions, not URL verbs           │
## │  • Consistent response format across all endpoints       │
## │  • Cursor pagination for real-time/social features       │
## │  • Version in URL path: /api/v1/...                     │
## │  • Stateless = scalable (token-based auth)              │
## │  • Max 2 levels of URL nesting                          │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **WSGI vs ASGI** — How Python web servers actually work under the hood